# non-diff-fn-wrap — ex3: chained non-diff ops — argmax(eq(a,b)) stays isolated

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `non-diff-fn-wrap`. Running the final beacon cell reports progress against the `Backprop: non-differentiable fn wrap` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: non-differentiable fn wrap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`non-diff-fn-wrap`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "non-diff-fn-wrap"
DD_SUBTOPIC = "Backprop: non-differentiable fn wrap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Non-diff op chained into another non-diff op — graph stays isolated

Ex1 wrapped a single non-diff op (`eq`); ex2 confirmed a non-diff branch is skipped during topo walk from a diff end-node. The deepening move chains TWO non-diff ops in series:

```
mask = eq(a, b)            # non-diff: requires_grad=False, recipe=None
idx  = argmax(mask)        # non-diff: same — operates on a non-tracked input
```

The interesting consequence: BOTH outputs have `recipe=None`. The graph walk from `idx` cannot reach `mask`, `a`, or `b` — it stops at `idx` itself (graph = `[idx]`). Each non-diff output is its own isolated leaf in the autograd-graph view, regardless of how many non-diff ops produced it.

**Why this is the right behaviour.** A chain of non-diff ops produces a tensor that cannot receive a meaningful grad — there's no chain rule that would let one flow. Treating it as a leaf in the graph is correct: the reverse pass simply has no work to do at it.

**Contrast with ex2.** Ex2 isolated ONE non-diff branch on a graph whose main path was differentiable. Here, EVERY op is non-diff — the entire subgraph collapses into independent single-node graphs.

### Exercise 3 — chained non-diff ops — argmax(eq(a,b)) stays isolated

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the cascade behaviour of chained non-differentiable ops: argmax(eq(a, b)) produces a MiniTensor whose recipe=None, requires_grad=False, and is unreachable from any other graph walk started from a different end-node.
> Keywords: non-diff, chain, argmax, eq, graph-isolation
> ```

**KCs targeted:** `non-diff-chain-isolation`, `recipe-none-cascades`

Implement `wrap_forward_fn(fwd_fn, is_differentiable=True)` (same contract as ex1/ex2: three-gate AND, conditional Recipe). Then use it to wrap TWO non-diff ops — `eq` and `argmax` — and build a compute graph that chains them.

Provided in the stub: `sorted_computational_graph(tensor)` walks the recipe-parents graph and stops at any `recipe=None` node.

Test inputs build (verbatim — your `wrap_forward_fn` must produce these structurally):
- `a`, `b`: tracked MiniTensor leaves.
- `mask = eq_wrap(a, b)` — first non-diff op. Recipe must be None.
- `idx  = argmax_wrap(mask)` — second non-diff op, OPERATES ON a non-tracked input. Recipe must also be None.
- `c = add_wrap(a, b)`, `d = mul_wrap(c, c)` — pure-diff side branch as a control.

The test verifies:
1. Both `mask.recipe is None` and `idx.recipe is None`.
2. Walking from `idx`: only `[idx]` is found (length 1 — chain stays isolated). `mask`, `a`, `b` are NOT in the walk.
3. Walking from `mask`: only `[mask]` (length 1).
4. Walking from `d`: includes `[d, c, a, b]` (the diff side); EXCLUDES `mask` and `idx`.
5. `mask.requires_grad is False` and `idx.requires_grad is False`.

In [ ]:
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        global_on = globals().get('grad_tracking_enabled', True)
        any_tracked = any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        requires_grad = bool(global_on and is_differentiable and any_tracked)
        out = MiniTensor(out_arr, requires_grad=requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(func=fwd_fn, args=raw_args, kwargs=kwargs, parents=parents)
        return out
    return tensor_func


<details><summary>Solution</summary>

```python
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        global_on = globals().get('grad_tracking_enabled', True)
        any_tracked = any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        requires_grad = bool(global_on and is_differentiable and any_tracked)
        out = MiniTensor(out_arr, requires_grad=requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(func=fwd_fn, args=raw_args, kwargs=kwargs, parents=parents)
        return out
    return tensor_func
```

**Recipe-None cascades.** When `eq_wrap(a, b)` produces `mask.requires_grad=False`, the subsequent `argmax_wrap(mask)` call sees `mask` as a non-tracked input. Even if `argmax` had been `is_differentiable=True`, the three-gate AND would still give `requires_grad=False` (no tracked input). Chaining preserves the non-tracking property.

**Per-non-diff-output is a graph singleton.** From the autograd graph's point of view, each non-diff output is its own isolated 1-node DAG. The semantic dependency `idx <- mask <- a, b` is INVISIBLE to the topo walker because there are no recipes encoding those edges. This is correct: no gradient flows along those edges, so the reverse pass has no work to do.

**Why this is a feature, not a limitation.** A user calling `loss.backward()` on a diff loss `d` that incidentally shares leaves with a non-diff `idx` doesn't want grad to flow through `idx`. The recipe=None mechanism makes that the default, with no conditional logic in the reverse pass.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()